# Tokenizer

## Load the text file

In [130]:
import os

with open("ProjectGutenberg.txt", 'r', encoding='utf-8') as f:
    sample_text = f.read()

# print(sample_text)
print(f"Number of characters: {len(sample_text)}")

Number of characters: 354046


## Use Regex to split words

In [131]:
import regex as re 

pattern = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

preprocessed_text = pattern.findall(sample_text)

print(f"Number of tokens: {len(preprocessed_text)}")
print(preprocessed_text[:30])

Number of tokens: 73773
['The', ' Project', ' Gutenberg', ' eBook', ' of', ' Glimpses', ' of', ' Japan', ' and', ' Formosa', '\n    ', '\n', 'This', ' eBook', ' is', ' for', ' the', ' use', ' of', ' anyone', ' anywhere', ' in', ' the', ' United', ' States', ' and', '\n', 'most', ' other', ' parts']


## Remove the whitespaces

In [132]:
preprocessed_text = [item.strip() for item in preprocessed_text if item.strip()]
# preprocessed_text

## Create the vocabulary

In [133]:
words=sorted(set(preprocessed_text))
print(words)
# print(len(words))

vocabulary={word:index for index,word in enumerate(words)}
# print(vocabulary)
print(len(vocabulary))

['!', '!)', '!—', '!”', '!”,', '#', '$', '%', '&', '(', '($', '(“', ')', ')(', '),', ');', '***', '+', ',', ',—', ',—“', ',”', '-', '-,', '------------------------------------------------------------------------', '-_', '-“', '.', '.)', '.,', '....', '....”', '...”', '.’”', '.”', '/', '000', '015', '07042', '1', '100', '101', '108', '109', '112', '113', '120', '121', '128', '129', '131', '132', '133', '136', '140', '141', '144', '145', '16', '160', '1600', '161', '1624', '1663', '17', '176', '177', '180', '181', '1859', '1863', '188', '1886', '189', '1895', '1896', '1899', '1905', '1913', '192', '1922', '1923', '1924', '193', '2', '20', '2001', '2026', '208', '209', '224', '225', '25', '250', '257', '272', '2753', '2999', '3', '30', '32', '33', '36', '37', '4', '41', '44', '45', '47', '48', '49', '5', '50', '500', '501', '516', '52', '53', '6', '60', '609', '61', '621', '6221541', '64', '65', '670', '7', '78', '78946', '8', '80', '81', '84', '842', '85', '862', '88', '89', '9', '90', '

# Tokenizer using BPE

## Converting text in utf-8

In [134]:
ids_list=[list(chunks.encode('utf-8')) for chunks in words]
print(ids_list)
print(len(ids_list))

[[33], [33, 41], [33, 226, 128, 148], [33, 226, 128, 157], [33, 226, 128, 157, 44], [35], [36], [37], [38], [40], [40, 36], [40, 226, 128, 156], [41], [41, 40], [41, 44], [41, 59], [42, 42, 42], [43], [44], [44, 226, 128, 148], [44, 226, 128, 148, 226, 128, 156], [44, 226, 128, 157], [45], [45, 44], [45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45], [45, 95], [45, 226, 128, 156], [46], [46, 41], [46, 44], [46, 46, 46, 46], [46, 46, 46, 46, 226, 128, 157], [46, 46, 46, 226, 128, 157], [46, 226, 128, 153, 226, 128, 157], [46, 226, 128, 157], [47], [48, 48, 48], [48, 49, 53], [48, 55, 48, 52, 50], [49], [49, 48, 48], [49, 48, 49], [49, 48, 56], [49, 48, 57], [49, 49, 50], [49, 49, 51], [49, 50, 48], [49, 50, 49], [49, 50, 56], [49, 50, 57], [49, 51, 49],

## Getting adjacent pairs

In [135]:
# Function to get adjacent pairs of IDs from a list of lists of IDs
def get_adjacent_pairs_count(ids_list):
    adjacent_pairs={}
    for ids in ids_list:
        for pair in zip(ids, ids[1:]):
            adjacent_pairs[pair]=adjacent_pairs.get(pair, 0) + 1

    return adjacent_pairs

# adjacent_pairs = get_adjacent_pairs_count(ids_list)
# print(adjacent_pairs)


## Merging two adjacent pairs

In [136]:
# Function to merge adjacent pairs into a new vocabulary and update the IDs list
def merge_adjacent_pairs(ids_list, adjacent_pairs, new_id):
    new_ids_list=[]
    for ids in ids_list:
        new_ids=[]
        i=0
        while i < len(ids):
            if i < len(ids)-1 and (ids[i], ids[i+1]) == adjacent_pairs:
                new_ids.append(new_id)
                i+=2
            else:
                new_ids.append(ids[i])
                i+=1
        new_ids_list.append(new_ids)
    return new_ids_list

# print("Before merging adjacent pairs:")
# print(ids_list)
# print("Adjacent pairs:")
# print(adjacent_pairs)
# print("After merging adjacent pairs:")
# new_ids_list = merge_adjacent_pairs(ids_list, adjacent_pairs)
# print(new_ids_list)

## Training Tokenizer

In [137]:
"""Create an encoder dictionary that maps each byte to its corresponding integer value.
utf-8 has 2**8 = 256 possible byte values, so we can create a dictionary that maps each byte to its corresponding integer value from 0 to 255.
This will be useful for encoding and decoding text data.
"""
encoder={bytes([i]):i for i in range(256)}

""" Creating hyperparameters for the tokenizer training.The vocabulary size is the number of unique tokens in the vocabulary,
    and the number of merges is the number of adjacent pairs that can be merged to create new tokens.
"""
vocab_size=len(vocabulary)
num_merges=vocab_size-2**8
current_idx=256

print(f"Vocabulary size: {vocab_size}")
print(f"Number of merges: {num_merges}")

# 1. Clean up the helper function so it works for ALL IDs
def get_bytes_for_id(idx, encoder_dict):
    # If it's a base byte, we can construct it instantly
    if idx < 256:
        return bytes([idx])
    # Otherwise, look it up dynamically from what we've built
    return next(k for k, v in encoder_dict.items() if v == idx)

# --- Inside your training loop ---

for k in range(num_merges):
    adjacent_pairs = get_adjacent_pairs_count(ids_list)
    if not adjacent_pairs:
        break
        
    most_frequent_pair = max(adjacent_pairs, key=adjacent_pairs.get)
    print(f"Iteration {k}: Most frequent pair: {most_frequent_pair}")

    # Correctly merge the sequence data
    ids_list = merge_adjacent_pairs(ids_list, most_frequent_pair, current_idx)

    # FIX: Retrieve raw bytes for both halves of the pair
    bytes_part1 = get_bytes_for_id(most_frequent_pair[0], encoder)
    bytes_part2 = get_bytes_for_id(most_frequent_pair[1], encoder)
    
    # FIX: Concatenate the raw byte sequences together
    byte_piece = bytes_part1 + bytes_part2
            
    # Store the new mapping (Bytes -> New ID)
    encoder[byte_piece] = current_idx
    current_idx += 1
    
# Rebuild the decoder at the very end of training
decoder = {v: k for k, v in encoder.items()}

Vocabulary size: 8788
Number of merges: 8532
Iteration 0: Most frequent pair: (105, 110)
Iteration 1: Most frequent pair: (101, 114)
Iteration 2: Most frequent pair: (101, 115)
Iteration 3: Most frequent pair: (101, 100)
Iteration 4: Most frequent pair: (111, 110)
Iteration 5: Most frequent pair: (256, 103)
Iteration 6: Most frequent pair: (101, 110)
Iteration 7: Most frequent pair: (97, 116)
Iteration 8: Most frequent pair: (97, 110)
Iteration 9: Most frequent pair: (97, 114)
Iteration 10: Most frequent pair: (111, 114)
Iteration 11: Most frequent pair: (115, 116)
Iteration 12: Most frequent pair: (114, 101)
Iteration 13: Most frequent pair: (97, 108)
Iteration 14: Most frequent pair: (105, 116)
Iteration 15: Most frequent pair: (108, 121)
Iteration 16: Most frequent pair: (111, 117)
Iteration 17: Most frequent pair: (105, 260)
Iteration 18: Most frequent pair: (105, 99)
Iteration 19: Most frequent pair: (105, 115)
Iteration 20: Most frequent pair: (108, 101)
Iteration 21: Most freque

KeyboardInterrupt: 

In [ ]:
print(list(encoder)[256:266])

[b'in', b'er', b'es', b'ed', b'on', b'ing', b'en', b'at', b'an', b'ar']


## Encoding

In [ ]:
def encode(text):
    text_chunks = pattern.findall(text)
    final_ids=[]

    for chunk in text_chunks:
        chunk_ids=list(chunk.encode('utf-8'))
        print(chunk_ids)

        while len(chunk_ids) >= 2:
            adjacent_pairs = get_adjacent_pairs(chunk_ids)
            pair=min(adjacent_pairs, key=adjacent_pairs.get)
            if pair not in encoder:
                break
                
        final_ids.extend(chunk_ids)
    return final_ids

print(encode("Hello, world! This is a test."))

[72, 101, 108, 108, 111]
[44]
[32, 119, 111, 114, 108, 100]
[33]
[32, 84, 104, 105, 115]
[32, 105, 115]
[32, 97]
[32, 116, 101, 115, 116]
[46]
[]
